# M49 · PointPillars KITTI LiDAR
Chọn GPU rồi Run all. Lần đầu biên dịch CUDA có thể mất vài phút.
Model/checkpoint: https://github.com/zhulf0804/PointPillars
Chỉ nhận point cloud và calibration, không đọc ground truth hoặc Manual.
Đánh giá car với tâm hộp trong FOV camera và vùng x [0,69.12), y [-39.68,39.68).
Đây là pilot M49, không phải KITTI AP hoặc phép đánh giá model trên tập giữ riêng.
Pipeline CUDA/PyTorch cần được kiểm chứng trong runtime Colab thực tế.


In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime → Change runtime type → GPU, rồi Run all. PointPillars cần CUDA.')
if subprocess.run(['which','nvcc'],capture_output=True).returncode:
    raise RuntimeError('Runtime cần CUDA toolkit (nvcc) để cài PointPillars.')
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
    'ninja==1.11.1.4','numba==0.61.2','open3d==0.19.0','opencv-python-headless==4.11.0.86','PyYAML==6.0.2'])
REPO = Path('/content/m49_pointpillars')
COMMIT = '620e6b0d07e4cb37b7b0114f26b934e8be92a0ba'
if not REPO.exists():
    subprocess.check_call(['git','init',str(REPO)])
    subprocess.check_call(['git','-C',str(REPO),'remote','add','origin','https://github.com/zhulf0804/PointPillars.git'])
    subprocess.check_call(['git','-C',str(REPO),'fetch','--depth','1','origin',COMMIT])
    subprocess.check_call(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'])
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() != COMMIT:
    raise RuntimeError('Thư mục PointPillars không đúng phiên bản; dùng runtime Colab mới.')
# Build against the CUDA/PyTorch supplied by Colab; do not install upstream's old Torch 1.8 requirements.
subprocess.check_call([sys.executable,'setup.py','build_ext','--inplace'],cwd=REPO,env={**os.environ,'MAX_JOBS':'2'})
sys.path.insert(0,str(REPO))
print('PointPillars ready:',COMMIT,'Torch:',torch.__version__,'CUDA:',torch.version.cuda)


In [ ]:
import json, time, hashlib, subprocess, sys
from pathlib import Path
import numpy as np
import torch
from pointpillars.model import PointPillars
"""Shared by generated notebook cells; stdlib only, testable without Colab."""
import hashlib
import json
from pathlib import Path


def load_drive_job(root, task):
    root = Path(root)
    request = json.loads((root/'request.json').read_text(encoding='utf-8'))
    if request['task'] != task or root.name != request['job_id']:
        raise ValueError('Sai notebook hoặc sai thư mục lượt chạy')
    names = request['manifest']
    if not isinstance(names, list) or not 1 <= len(names) <= 20 or len(names) != len(set(names)):
        raise ValueError('Manifest không hợp lệ')
    for name in names:
        if not isinstance(name, str) or not name or '/' in name or '\\' in name or ':' in name or name in {'.', '..'}:
            raise ValueError('Tên mẫu không hợp lệ')
        path = root/'assets'/name
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != request['sha256'].get(name):
            raise ValueError(f'{name}: chờ Drive đồng bộ đủ dữ liệu rồi chạy lại')
    return request


def publish_drive_result(root, request, bundle, timing, environment):
    root = Path(root)
    if root.name != request['job_id'] or bundle['task'] != request['task']:
        raise ValueError('Sai lượt chạy hoặc tác vụ')
    if sorted(s['asset_name'] for s in bundle['samples']) != sorted(request['manifest']):
        raise ValueError('Kết quả thiếu/trùng/thừa mẫu')
    if (root/'result.ready.json').exists():
        raise ValueError('Lượt này đã xuất kết quả. Tạo lượt mới trong ứng dụng để chạy lại.')
    raw = json.dumps(bundle, ensure_ascii=False, allow_nan=False, indent=2).encode('utf-8')
    if len(raw) > 32*1024*1024:
        raise ValueError('Kết quả vượt 32 MiB')
    outputs = {'predictions.json': raw,
               'timing.json': json.dumps(timing, ensure_ascii=False, allow_nan=False, indent=2).encode('utf-8'),
               'environment.txt': environment.encode('utf-8')}
    for name, data in outputs.items():
        temporary = root/(name+'.tmp')
        temporary.write_bytes(data)
        temporary.replace(root/name)
    receipt = {'job_id': request['job_id'], 'sha256': hashlib.sha256(raw).hexdigest()}
    temporary = root/'result.ready.tmp'
    temporary.write_text(json.dumps(receipt), encoding='utf-8')
    temporary.replace(root/'result.ready.json')
    return receipt

from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/M49/PASTE_JOB_ID')
request = load_drive_job(ROOT, 'detection_3d')
assert request['classes'] == ['car'], 'Pipeline này dùng lớp car của KITTI'
model_spec = request['model']
assert model_spec.get('pipeline') == 'pointpillars_kitti', 'Sai pipeline 3D'
assert set(request['inputs']) == set(request['manifest']), 'Thiếu calibration theo mẫu'
CHECKPOINT = REPO/'pretrained/epoch_160.pth'
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() == '482dfcf63b932cc5ccf012b4bbdad52aa51aa33becf87d0a39d61c39b377b5b1', 'Checkpoint không đúng bản đã khóa'
CONFIDENCE = model_spec['confidence']
model = PointPillars(nclasses=3).cuda()
model.load_state_dict(torch.load(CHECKPOINT,map_location='cpu',weights_only=True))
model.score_thr = min(0.1,CONFIDENCE)
model.eval()

"""PointPillars (zhulf0804) output convention -> M49; also embedded in Colab.

Upstream boxes are [x,y,bottom_z,width,length,height,rotation_y]. Its corner
routine rotates local coordinates clockwise; length lies on local Y.
M49 uses geometric center, [length,width,height], counterclockwise length yaw.
"""
import math


def pointpillars_geometry(box):
    if len(box) != 7 or not all(math.isfinite(float(v)) for v in box):
        raise ValueError('PointPillars box must contain seven finite values')
    x,y,z,width,length,height,angle = map(float,box)
    if min(width,length,height) <= 0:
        raise ValueError('PointPillars box dimensions must be positive')
    yaw = (math.pi/2-angle+math.pi) % (2*math.pi)-math.pi
    return dict(kind='cuboid3d',center=[x,y,z+height/2],size_lwh=[length,width,height],yaw=yaw,frame='lidar')


def in_evaluation_scope(geometry, camera):
    """Same center-based forward range and camera FOV for GT and predictions."""
    x,y,z = geometry['center']
    if not (0 <= x < 69.12 and -39.68 <= y < 39.68):
        return False
    p = [x,y,z,1]
    cam = [sum(row[j]*p[j] for j in range(4)) for row in camera['Tr_velo_to_cam']]
    rect = [sum(row[j]*cam[j] for j in range(3)) for row in camera['R0_rect']]+[1]
    projected = [sum(row[j]*rect[j] for j in range(4)) for row in camera['P2']]
    h,w = camera['image_shape']
    return projected[2] > 0 and 0 <= projected[0]/projected[2] < w and 0 <= projected[1]/projected[2] < h


def pointpillars_objects(result, confidence, camera):
    boxes, labels, scores = result['lidar_bboxes'], result['labels'], result['scores']
    if len(boxes) != len(labels) or len(labels) != len(scores):
        raise ValueError('PointPillars output lengths differ')
    objects = []
    for index, (box, label, score) in enumerate(zip(boxes,labels,scores)):
        score = float(score)
        if not math.isfinite(score) or not 0 <= score <= 1:
            raise ValueError('Invalid confidence from PointPillars')
        if int(label) != 2 or score < confidence:  # Upstream label 2 = Car.
            continue
        geometry = pointpillars_geometry(box)
        if in_evaluation_scope(geometry,camera):
            objects.append(dict(id=str(index),category='car',confidence=score,geometry=geometry))
    return objects


In [ ]:
def stamp():
    torch.cuda.synchronize()
    return time.perf_counter()

samples, per_sample = [], []
for name in request['manifest']:
    # File I/O and network/model setup are outside the measured inference interval.
    points = np.fromfile(ROOT/'assets'/name,dtype='<f4')
    if not points.size or points.size % 4:
        raise ValueError('LiDAR phải là BIN x,y,z,intensity không rỗng')
    points = points.reshape(-1,4)
    if not np.isfinite(points).all():
        raise ValueError('LiDAR chứa NaN/Inf')
    started = stamp()
    lower = np.array([0,-39.68,-3],dtype=np.float32)
    upper = np.array([69.12,39.68,1],dtype=np.float32)
    points = points[((points[:,:3] > lower) & (points[:,:3] < upper)).all(axis=1)]
    if not len(points):
        raise ValueError('Không có điểm trong phạm vi PointPillars; không xuất dự đoán giả')
    with torch.inference_mode():
        result = model(batched_pts=[torch.from_numpy(points).cuda()],mode='test')[0]
    # Upstream returns [] if no anchor passes its score threshold.
    objects = [] if isinstance(result,list) and not result else pointpillars_objects(result,CONFIDENCE,request['inputs'][name])
    seconds = stamp()-started
    samples.append(dict(asset_name=name,objects=objects))
    per_sample.append(dict(asset_name=name,inference_seconds=seconds))
    print(name,len(objects),'cars',round(seconds,3),'s')
INFERENCE_SECONDS = sum(row['inference_seconds'] for row in per_sample)


In [ ]:
bundle = dict(schema_version='1.0',task='detection_3d',name='PointPillars KITTI',
    run=dict(model='PointPillars (zhulf0804)',checkpoint='epoch_160.pth sha256:482dfcf63b932cc5ccf012b4bbdad52aa51aa33becf87d0a39d61c39b377b5b1',
             gpu=torch.cuda.get_device_name(0),library_version='PointPillars 620e6b0d07e4cb37b7b0114f26b934e8be92a0ba; torch '+torch.__version__,
             inference_seconds=INFERENCE_SECONDS),samples=samples)
timing = dict(inference_seconds=INFERENCE_SECONDS,per_sample=per_sample,
              includes='point range filter, CPU-GPU transfer, model, NMS, conversion/FOV filter; excludes download, installation, file IO')
environment = subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
publish_drive_result(ROOT,request,bundle,timing,environment)
print('Đã gửi kết quả 3D. Quay lại M49 để sửa nhãn AI.')
